# Example 03 — Switch models

Same toy regression as example 01. You change **one** YAML key: `modeling.model_selection` (default here: `randomforest`).

## Learning objectives

- Swap ExtraTrees / Random Forest / Ridge / … without rewriting training code.
- See the architecture in the saved `models/*.pkl` filename.

```bash
python -m run_pipeline --config examples/03_switch_models/config.yaml --force-retrain
```

[Open in Colab](https://colab.research.google.com/github/Oikiy/ML-MEM-Review/blob/main/examples/03_switch_models/Colab.ipynb)


## 1. Bootstrap (Colab clone + install)

**What you will do.** Run the next cell so this notebook can see the full `ML-MEM-Review` repo.

**Why it matters.** The example YAML, CSV, and `src/` are not inside this `.ipynb`. On Colab the cell clones or updates `/content/ML-MEM-Review` and pip-installs the package. Locally it only finds the repo (example 05 will install Ax if needed).

**What to look for.** `Repo root: ...` and `Colab bootstrap done.` First Colab install can take a few minutes. No Google Drive.

**Common pitfall.** Running later cells first. Expand the hidden **Bootstrap** form if clone/install fails.

Run cells in order after this.


In [ ]:
# @title Bootstrap (clone + install)
# === COLAB BOOTSTRAP (safe locally) =========================================
import os
import sys
import subprocess
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _run(cmd, **kwargs):
    print("+", " ".join(cmd) if isinstance(cmd, (list, tuple)) else cmd)
    subprocess.check_call(cmd, **kwargs)

def _find_repo_root(start: Path):
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").is_file() and (candidate / "src").is_dir():
            return candidate
    return None

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

IN_COLAB = _in_colab()
REPO_URL = os.environ.get("ML_MEM_REPO_URL", "https://github.com/Oikiy/ML-MEM-Review.git")
REPO_NAME = os.environ.get("ML_MEM_REPO_NAME", "ML-MEM-Review")
REPO_REF = os.environ.get("ML_MEM_REPO_REF", "main")
COLAB_ROOT = Path(os.environ.get("ML_MEM_COLAB_ROOT", f"/content/{REPO_NAME}"))
EXTRAS = ''

def _git_head(repo: Path) -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError, OSError):
        return ""

def _truthy(name: str) -> bool:
    return os.environ.get(name, "").strip().lower() in {"1", "true", "yes"}

if IN_COLAB:
    try:
        if COLAB_ROOT.exists() and (COLAB_ROOT / ".git").is_dir():
            if _truthy("ML_MEM_SKIP_PULL"):
                print(f"Skipping git update at {COLAB_ROOT} (ML_MEM_SKIP_PULL=1)")
            else:
                print(f"Updating clone at {COLAB_ROOT} to origin/{REPO_REF}")
                _run(["git", "-C", str(COLAB_ROOT), "fetch", "--depth", "1", "origin", REPO_REF])
                _run(["git", "-C", str(COLAB_ROOT), "reset", "--hard", "FETCH_HEAD"])
        elif not COLAB_ROOT.exists():
            _run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(COLAB_ROOT)])
        os.chdir(COLAB_ROOT)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "git clone/update failed. Make the repo public, set ML_MEM_REPO_URL "
            f"to a fork, or set ML_MEM_REPO_REF. Original error: {exc}"
        ) from exc
    root = _find_repo_root(Path.cwd())
else:
    root = _find_repo_root(Path.cwd())

if root is None:
    raise RuntimeError(
        "Could not locate ML-MEM-Review (need config.yaml + src/). "
        "On Colab: re-run this cell after a successful clone. "
        "Locally: open from inside a full clone of the repository."
    )

os.chdir(root)
print(f"Repo root: {root}  git={_git_head(root)[:12]}")

src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB:
    marker = root / ".colab_install_ok"
    force = _truthy("ML_MEM_FORCE_INSTALL")
    head = _git_head(root)
    extras = EXTRAS
    pip_spec = str(root) if not extras else f"{root}[{extras}]"
    install_key = f"{head}\n{extras}"
    prev = marker.read_text(encoding="utf-8").strip() if marker.is_file() else ""
    if force or prev != install_key:
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", pip_spec])
        try:
            marker.write_text(install_key + "\n", encoding="utf-8")
        except OSError:
            pass
    else:
        print("Skipping pip install (clone SHA + extras unchanged). Set ML_MEM_FORCE_INSTALL=1 to reinstall.")
    print("Colab bootstrap done.")
else:
    print("Local session: repo root + src/ on path (no auto pip install).")

REPO_ROOT = root
# === end bootstrap ===========================================================


## What this config emphasizes

- **The knob:** `modeling.model_selection` (here `randomforest`).
- Same toy regression CSV as example 01.
- Allowed names include `extratrees`, `randomforest`, `ridge`, `elasticnet`, `linear`, `histgradientboosting`.

**Try it yourself.** After the first run, change that one key and re-run with `--force-retrain`. Check the new filename in `models/`.


## 2. Read `config.yaml`

**What you will do.** Print this example’s YAML. Nothing is trained yet.

**Why it matters.** This gallery teaches *configuration*, not new Python. The keys you change here are how you reuse `run_pipeline` on your own table.

**What to look for.** The settings listed in “What this config emphasizes” above.

**Try it yourself.** Before running the pipeline, name the target column you expect the model to predict.


In [ ]:
# Show the example config (edit the file under examples/ to experiment)
from pathlib import Path

CONFIG_REL = "examples/03_switch_models/config.yaml"
CONFIG_PATH = REPO_ROOT / CONFIG_REL
assert CONFIG_PATH.is_file(), f"Missing {CONFIG_PATH} — bootstrap/clone incomplete?"

print(CONFIG_PATH.read_text(encoding="utf-8"))



## 3. Run the pipeline

**What you will do.** The next cell calls `python -m run_pipeline` for this example, one **stage** at a time (`eda, then model`).

**Why it matters.** Same command you would run in a terminal. Results are written under `examples/03_switch_models/`, not into the notebook file.

**What to look for.** Printed `Pipeline finished` (and for example 05, inverse-design trial counts). On Colab, open the `examples/03_switch_models/outputs/figures` folder in the left file browser.

**Common pitfall.** The red `RuntimeError` at the bottom is only a wrapper. Read the traceback **above** it. Run Bootstrap and this cell in order. `--force-retrain` rebuilds models if you edited YAML.


In [ ]:
# Run the pipeline (artifacts written under examples/03_switch_models/)
import os
import subprocess
import sys
from pathlib import Path

env = os.environ.copy()
src = str(Path(REPO_ROOT) / "src")
env["PYTHONPATH"] = src + os.pathsep + env.get("PYTHONPATH", "")
base = [sys.executable, "-m", "run_pipeline", "--config", str(CONFIG_PATH), "--force-retrain"]
for stage in ("eda", "model"):
    cmd = base + ["--stage", stage]
    print("+", " ".join(cmd))
    completed = subprocess.run(
        cmd, cwd=str(REPO_ROOT), env=env, text=True, capture_output=True
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        err = (completed.stderr or completed.stdout or "").strip()
        raise RuntimeError(
            f"run_pipeline --stage {stage} exited {completed.returncode}.\n\n{err[-8000:]}"
        )
print("Done.")



## 4. Show figures

**What you will do.** Open the PNGs this example just wrote, in workflow order.

**What to look for.** Order: distributions → heatmap → parity.

**Common pitfall.** Empty or missing files ⇒ the pipeline cell did not finish. Re-run §3.


In [ ]:
# @title Show figures
from pathlib import Path
from IPython.display import display, Markdown, Image

fig_dir = REPO_ROOT / "examples" / "03_switch_models" / "outputs" / "figures"
for name in [
    "target_distributions.png",
    "pearson_heatmap.png",
    "parity_permeance.png",
]:
    path = fig_dir / name
    if path.is_file():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(path)))


## Try next

- Set `model_selection: ridge` and compare.
- Example 04: choose columns, not estimators.
- Full tutorial: [`Ml-MEM-Tut.ipynb`](../../notebooks/Complete_notebook/Ml-MEM-Tut.ipynb)
